In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score, roc_curve
from sklearn.utils.class_weight import compute_class_weight

np.random.seed(42)

def create_credit_card_dataset(n_samples=10000):
    data = {
        'Customer_Age': np.random.randint(18, 80, n_samples),
        'Gender': np.random.choice(['M', 'F'], n_samples),
        'Dependent_count': np.random.randint(0, 6, n_samples),
        'Education_Level': np.random.choice(['High School', 'Graduate', 'Uneducated', 'College', 'Post-Graduate', 'Doctorate'], n_samples),
        'Marital_Status': np.random.choice(['Married', 'Single', 'Divorced'], n_samples),
        'Income_Category': np.random.choice(['Less than $40K', '$40K - $60K', '$60K - $80K', '$80K - $120K', '$120K +'], n_samples),
        'Card_Category': np.random.choice(['Blue', 'Silver', 'Gold', 'Platinum'], n_samples),
        'Months_on_book': np.random.randint(13, 57, n_samples),
        'Total_Relationship_Count': np.random.randint(1, 7, n_samples),
        'Months_Inactive_12_mon': np.random.randint(0, 7, n_samples),
        'Contacts_Count_12_mon': np.random.randint(0, 7, n_samples),
        'Credit_Limit': np.random.uniform(1400, 35000, n_samples),
        'Total_Revolving_Bal': np.random.uniform(0, 2600, n_samples),
        'Avg_Open_To_Buy': np.random.uniform(0, 35000, n_samples),
        'Total_Amt_Chng_Q4_Q1': np.random.uniform(0, 3.4, n_samples),
        'Total_Trans_Amt': np.random.uniform(500, 18500, n_samples),
        'Total_Trans_Ct': np.random.randint(10, 140, n_samples),
        'Total_Ct_Chng_Q4_Q1': np.random.uniform(0, 3.8, n_samples),
        'Avg_Utilization_Ratio': np.random.uniform(0, 1, n_samples)
    }


    churn_prob = (
        (data['Months_Inactive_12_mon'] > 3) * 0.3 +
        (data['Total_Trans_Ct'] < 30) * 0.25 +
        (data['Contacts_Count_12_mon'] > 4) * 0.2 +
        (data['Total_Relationship_Count'] < 3) * 0.15 +
        (data['Avg_Utilization_Ratio'] < 0.1) * 0.1
    )

    data['Attrition_Flag'] = np.random.binomial(1, churn_prob, n_samples)
    return pd.DataFrame(data)


df = create_credit_card_dataset()
df['Attrition_Flag'] = df['Attrition_Flag'].map({0: 'Existing Customer', 1: 'Attrited Customer'})

print(f"Dataset created successfully!")
print(f"Shape: {df.shape}")
print(f"Features: {list(df.columns)}")

Credit Card Customer Churn Prediction Analysis
Dataset created successfully!
Shape: (10000, 20)
Features: ['Customer_Age', 'Gender', 'Dependent_count', 'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category', 'Months_on_book', 'Total_Relationship_Count', 'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 'Credit_Limit', 'Total_Revolving_Bal', 'Avg_Open_To_Buy', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt', 'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio', 'Attrition_Flag']


In [2]:

print(f"Dataset Shape: {df.shape}")
print(f"Missing Values: {df.isnull().sum().sum()}")


print("\nTarget Variable Distribution:")
target_dist = df['Attrition_Flag'].value_counts()
print(target_dist)
print(f"Percentages:")
print(df['Attrition_Flag'].value_counts(normalize=True) * 100)


print("\nBasic Statistics for Numerical Features:")
numerical_cols = df.select_dtypes(include=[np.number]).columns
print(df[numerical_cols].describe().round(2))


print("\nCategorical Features Summary:")
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if col != 'Attrition_Flag':
        print(f"{col}: {df[col].nunique()} unique values")

print("EDA completed successfully!")

majority_class = df['Attrition_Flag'].value_counts().max()
minority_class = df['Attrition_Flag'].value_counts().min()
imbalance_ratio = majority_class / minority_class

print(f"Imbalance Ratio: {imbalance_ratio:.2f}")
if imbalance_ratio > 1.5:
    print(" Data is imbalanced - will handle using class weights")
    imbalanced = True
else:
    print(" Data is relatively balanced")
    imbalanced = False


1. EXPLORATORY DATA ANALYSIS
Dataset Shape: (10000, 20)
Missing Values: 0

Target Variable Distribution:
Attrition_Flag
Existing Customer    7131
Attrited Customer    2869
Name: count, dtype: int64
Percentages:
Attrition_Flag
Existing Customer    71.31
Attrited Customer    28.69
Name: proportion, dtype: float64

Basic Statistics for Numerical Features:
       Customer_Age  Dependent_count  Months_on_book  \
count       10000.0         10000.00        10000.00   
mean           48.8             2.48           34.54   
std            17.9             1.72           12.68   
min            18.0             0.00           13.00   
25%            34.0             1.00           24.00   
50%            49.0             2.00           35.00   
75%            64.0             4.00           46.00   
max            79.0             5.00           56.00   

       Total_Relationship_Count  Months_Inactive_12_mon  \
count                  10000.00                10000.00   
mean                 

In [3]:

X = df.drop('Attrition_Flag', axis=1).copy()
y = df['Attrition_Flag'].copy()


categorical_cols = ['Gender', 'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category']
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le

print(" Categorical variables encoded")


le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)
print(" Target variable encoded (0: Existing, 1: Attrited)")


X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f" Data split - Train: {X_train.shape}, Test: {X_test.shape}")


class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(zip(np.unique(y_train), class_weights))
print(f" Class weights calculated: {class_weight_dict}")


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Features scaled using StandardScaler")

print("Data preprocessing completed successfully!")


results = {}

def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)

    if hasattr(model, 'predict_proba'):
        train_prob = model.predict_proba(X_train)[:, 1]
        test_prob = model.predict_proba(X_test)[:, 1]
        train_auc = roc_auc_score(y_train, train_prob)
        test_auc = roc_auc_score(y_test, test_prob)
    else:
        train_auc = test_auc = None

    return {
        'model': model,
        'train_acc': train_acc,
        'test_acc': test_acc,
        'train_auc': train_auc,
        'test_auc': test_auc,
        'test_pred': test_pred
    }


3. DATA PREPROCESSING
 Categorical variables encoded
 Target variable encoded (0: Existing, 1: Attrited)
 Data split - Train: (8000, 19), Test: (2000, 19)
 Class weights calculated: {np.int64(0): np.float64(1.7429193899782136), np.int64(1): np.float64(0.7011393514461)}
Features scaled using StandardScaler
Data preprocessing completed successfully!

4. MACHINE LEARNING MODELS


In [4]:

# Model 1: Logistic Regression
print("\n4.1 Logistic Regression")
lr_model = LogisticRegression(random_state=42, max_iter=500)
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)
lr_acc = accuracy_score(y_test, lr_pred)
lr_prob = lr_model.predict_proba(X_test_scaled)[:, 1]
lr_auc = roc_auc_score(y_test, lr_prob)
print(f" Test Accuracy: {lr_acc:.4f}, AUC: {lr_auc:.4f}")

# Model 2: Naive Bayes
print("\n4.2 Naive Bayes")
nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)
nb_pred = nb_model.predict(X_test_scaled)
nb_acc = accuracy_score(y_test, nb_pred)
nb_prob = nb_model.predict_proba(X_test_scaled)[:, 1]
nb_auc = roc_auc_score(y_test, nb_prob)
print(f" Test Accuracy: {nb_acc:.4f}, AUC: {nb_auc:.4f}")

# Model 3: Decision Tree
print("\n4.3 Decision Tree")
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train_scaled, y_train)
dt_pred = dt_model.predict(X_test_scaled)
dt_acc = accuracy_score(y_test, dt_pred)
dt_prob = dt_model.predict_proba(X_test_scaled)[:, 1]
dt_auc = roc_auc_score(y_test, dt_prob)
print(f" Test Accuracy: {dt_acc:.4f}, AUC: {dt_auc:.4f}")

# Model 4: Random Forest
print("\n4.4 Random Forest")
rf_model = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)
rf_acc = accuracy_score(y_test, rf_pred)
rf_prob = rf_model.predict_proba(X_test_scaled)[:, 1]
rf_auc = roc_auc_score(y_test, rf_prob)
print(f" Test Accuracy: {rf_acc:.4f}, AUC: {rf_auc:.4f}")


model_results = {
    'Model': ['Logistic Regression', 'Naive Bayes', 'Decision Tree', 'Random Forest'],
    'Test_Accuracy': [lr_acc, nb_acc, dt_acc, rf_acc],
    'Test_AUC': [lr_auc, nb_auc, dt_auc, rf_auc]
}

results_df = pd.DataFrame(model_results)
print(f"\nAll models trained successfully!")

print(results_df.to_string(index=False))


best_model_idx = results_df['Test_AUC'].idxmax()
best_model_name = results_df.loc[best_model_idx, 'Model']
best_auc = results_df.loc[best_model_idx, 'Test_AUC']

print(f"\n Best Model: {best_model_name}")
print(f" Best AUC Score: {best_auc:.4f}")

if best_model_name == 'Logistic Regression':
    best_model_obj = lr_model
    best_pred = lr_pred
elif best_model_name == 'Naive Bayes':
    best_model_obj = nb_model
    best_pred = nb_pred
elif best_model_name == 'Decision Tree':
    best_model_obj = dt_model
    best_pred = dt_pred
else:
    best_model_obj = rf_model
    best_pred = rf_pred


4.1 Logistic Regression
 Test Accuracy: 0.7195, AUC: 0.7047

4.2 Naive Bayes
 Test Accuracy: 0.7220, AUC: 0.7156

4.3 Decision Tree
 Test Accuracy: 0.7335, AUC: 0.7593

4.4 Random Forest
 Test Accuracy: 0.7305, AUC: 0.7665

All models trained successfully!
              Model  Test_Accuracy  Test_AUC
Logistic Regression         0.7195  0.704699
        Naive Bayes         0.7220  0.715609
      Decision Tree         0.7335  0.759267
      Random Forest         0.7305  0.766516

🏆 Best Model: Random Forest
🏆 Best AUC Score: 0.7665
